In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("Could not find repo root (missing pyproject.toml).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("python:", sys.executable)
if ".venv" not in sys.executable:
    print("WARNING: use the .venv kernel / interpreter (expected .venv/bin/python)")

from src.train.prott5_go_train import TrainConfig, train
from src.test.prott5_go_predict import predict

In [ ]:
data_path = REPO_ROOT / "data"
train_fasta = data_path / "Train" / "train_sequences.fasta"
train_terms = data_path / "Train" / "train_terms.tsv"
obo_path = data_path / "Train" / "go-basic.obo"
test_fasta = data_path / "Test" / "testsuperset.fasta"

model_ckpt = "Rostlab/prot_t5_xl_uniref50"
model_dir = REPO_ROOT / "models" / "prott5-go-mf"
pred_tsv = REPO_ROOT / "pred.tsv"

In [ ]:
# Training (set DO_TRAIN=True to run)
DO_TRAIN = False

if DO_TRAIN:
    cfg = TrainConfig(
        train_fasta=train_fasta,
        train_terms_tsv=train_terms,
        output_dir=model_dir,
        ckpt_name=model_ckpt,
        namespace="MF",  # or: "BP", "CC", or None
        min_label_count=5,
        max_labels=4096,
        max_length=1024,
        train_batch_size=1,
        num_train_epochs=1,
        learning_rate=2e-4,
        freeze_encoder=False,
        unfreeze_last_n_layers=2,
        val_fraction=0.01,
        seed=0,
    )
    train(cfg)

In [ ]:
# Prediction
if not model_dir.exists():
    raise FileNotFoundError(f"Missing model_dir={model_dir}. Train first (or point model_dir at a trained model).")

predict(
    model_dir=model_dir,
    test_fasta=test_fasta,
    out_tsv=pred_tsv,
    batch_size=1,
    top_k=1500,
    min_score=0.001,
    max_terms_per_protein=1500,
    obo_path=obo_path,
    propagate=True,
)
pred_tsv